# Stochastic Oversold Reversal on SPY
## Strategy Brief
The Stochastic Oversold Reversal strategy aims to capitalize on mean reversion by identifying oversold conditions in the SPY ETF using the Stochastic Oscillator. When the Stochastic Oscillator drops below a certain threshold, it signals a potential reversal, indicating a buying opportunity. The strategy exits the position when the Stochastic Oscillator indicates a return to neutral or overbought conditions. Historically, this strategy has shown potential for capturing short-term reversals in SPY's price movements.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for the Stochastic Oversold Reversal strategy. These include the lookback period for the Stochastic Oscillator, the oversold threshold, and the overbought threshold.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
STOCH_PERIOD = 14
OVERSOLD_THRESHOLD = 20
OVERBOUGHT_THRESHOLD = 80

## PHASE 2 - Data Exploration
We will download SPY data using yfinance from January 1, 2010, to today. Then, we will compute the Stochastic Oscillator and plot it overlaid on the SPY price.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate Stochastic Oscillator
data['Low_14'] = data['Low'].rolling(window=STOCH_PERIOD).min()
data['High_14'] = data['High'].rolling(window=STOCH_PERIOD).max()
data['%K'] = 100 * ((data['Close'] - data['Low_14']) / (data['High_14'] - data['Low_14']))

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['%K'], label='Stochastic %K', linestyle='--')
plt.axhline(y=OVERSOLD_THRESHOLD, color='r', linestyle='--', label='Oversold Threshold')
plt.axhline(y=OVERBOUGHT_THRESHOLD, color='g', linestyle='--', label='Overbought Threshold')
plt.title('SPY Price and Stochastic Oscillator')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We will create a signal series based on the Stochastic Oscillator. A buy signal is generated when the %K line crosses above the oversold threshold, and a sell signal is generated when it crosses below the overbought threshold.

In [ ]:
data['Signal'] = 0

data.loc[data['%K'] < OVERSOLD_THRESHOLD, 'Signal'] = 1  # Buy signal

data.loc[data['%K'] > OVERBOUGHT_THRESHOLD, 'Signal'] = -1  # Sell signal

# Forward fill the signals
data['Signal'] = data['Signal'].replace(to_replace=0, method='ffill')

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by shifting the positions by one day to avoid lookahead bias. We will then calculate daily returns and plot the equity curve.

In [ ]:
data['Position'] = data['Signal'].shift(1)
data['Daily_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Position'] * data['Daily_Return']
data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.title('Equity Curve of Stochastic Oversold Reversal Strategy')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We will calculate performance metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown. We will compare the strategy's performance against a buy-and-hold strategy.

In [ ]:
def calculate_performance(data):
    # CAGR
    n_years = (data.index[-1] - data.index[0]).days / 365.25
    cagr = (data['Equity_Curve'].iloc[-1]) ** (1 / n_years) - 1
    
    # Sharpe Ratio
    sharpe_ratio = data['Strategy_Return'].mean() / data['Strategy_Return'].std() * np.sqrt(252)
    
    # Sortino Ratio
    downside_std = data[data['Strategy_Return'] < 0]['Strategy_Return'].std()
    sortino_ratio = data['Strategy_Return'].mean() / downside_std * np.sqrt(252)
    
    # Calmar Ratio
    max_drawdown = (data['Equity_Curve'].cummax() - data['Equity_Curve']).max()
    calmar_ratio = cagr / max_drawdown
    
    # Buy and Hold
    buy_hold_cagr = (data['Close'].iloc[-1] / data['Close'].iloc[0]) ** (1 / n_years) - 1
    
    return {
        'CAGR': cagr,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Calmar Ratio': calmar_ratio,
        'Max Drawdown': max_drawdown,
        'Buy and Hold CAGR': buy_hold_cagr
    }

performance = calculate_performance(data)
performance_df = pd.DataFrame([performance])
print(performance_df)

## PHASE 6 - Deploy & Monitor
We will create a function that downloads the last 60 days of SPY data, computes today's Stochastic Oscillator signal, and prints the suggested position.

In [ ]:
def get_latest_signal():
    latest_data = yf.download('SPY', period='60d')
    latest_data['Low_14'] = latest_data['Low'].rolling(window=STOCH_PERIOD).min()
    latest_data['High_14'] = latest_data['High'].rolling(window=STOCH_PERIOD).max()
    latest_data['%K'] = 100 * ((latest_data['Close'] - latest_data['Low_14']) / (latest_data['High_14'] - latest_data['Low_14']))
    latest_signal = 0
    if latest_data['%K'].iloc[-1] < OVERSOLD_THRESHOLD:
        latest_signal = 1  # Buy
    elif latest_data['%K'].iloc[-1] > OVERBOUGHT_THRESHOLD:
        latest_signal = -1  # Sell
    print(f"Today's signal: {latest_signal}")

get_latest_signal()